# 04 — Gold: Fact Tables

**Phase:** 6 (Mon 18 May evening – Wed 20 May)

Builds 14 fact tables in `NEXAMART_GOLD`. Each owner writes:
1. **Grain declaration in their own words** (markdown cell) — graded heavily
2. Implementation cell respecting the declared grain
3. Verification cell asserting `COUNT(*) = COUNT(DISTINCT <grain key combo>)`

## Universal rules (lead enforces in PR review)

1. **Surrogate-key FKs only.** Never operational natural keys.
2. **`metric_certainty_level` populated** on every fact row.
3. **Don't mix grains.** One row = one atomic event/snapshot/state at the declared grain.
4. **Don't SUM non-additive facts** (`unit_price`, `margin_percentage`, `seller_risk_score`, `identity_confidence_score`). Store and let marts aggregate correctly.
5. **Semi-additive ATP** is valid across SKU/location on one date, INVALID across dates. Note in fact comment.
6. **Degenerate dimensions** (receipt_no, EC order_no, return auth, NL listing ref, canonical_case_key) stay as columns IN the fact, not as separate dim FKs.
7. **Rate metrics in seller fact:** store numerator + denominator separately, NOT pre-divided ratios.

## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports & helpers

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

def assert_grain(df, keys, name):
    """Grain check: COUNT(*) must equal COUNT(DISTINCT keys)."""
    total = df.count()
    distinct = df.select(*keys).distinct().count()
    ok = (total == distinct)
    print(f"[grain] {name}: rows={total}  distinct({', '.join(keys)})={distinct}  -> {'PASS' if ok else 'FAIL'}")
    return ok

# date_key matches dim_date (surrogate_key over the ISO date string).
def date_key(col):
    return surrogate_key(col)

# NOTE (member-dim FKs): M2/M3/M4/M6 dims (customer/product/store/seller) are not yet built.
# We populate their FKs with surrogate_key(natural_code) — the SAME convention those dims use — so the
# joins resolve once members deliver; the natural key is also kept on the fact for traceability.

---
# Sales Group (M5)

## fact_store_sale_line (M5)

### Grain declaration

> **TODO M5: write in your own words. Required.**
> Suggested starting point: "One row in `fact_store_sale_line` represents a single product line item scanned on a POS receipt. The grain is enforced by (`receipt_no`, `line_no`)."

### Dim FKs to join
- date_key (txn_date), product_key, store_key, customer_key, payment_method_key, channel_key (=STORE), promotion_key

### Degenerate dims (stay in fact)
- `receipt_no`, `line_no`

### Additive measures
- `quantity`, `gross_amount`, `discount_amount`, `net_amount`, `tax_amount`, `cogs_amount`

### Non-additive (store, but warn marts)
- `unit_price`, `margin_percentage`

### Metric certainty
- `metric_certainty_level` = CONFIRMED (POS is a recorded transaction)

In [ ]:
# TODO M5: implement; then assert_grain(fact, ['receipt_no', 'line_no'], 'fact_store_sale_line')

## fact_ecommerce_order_line (Lead)

### Grain declaration

> One row represents a single product line on an EC order. Grain enforced by (`order_id`, `line_id`). Mixing in shipment-fulfilment events would break additivity for `quantity`, since one line can be split across multiple shipments — that is `fact_order_fulfilment`'s job.

### Dim FKs
- date_key (order_date), product_key, customer_key, promotion_key, channel_key (=EC or BOPIS), delivery_method_key, store_key (BOPIS pickup store, NULL for home delivery)

### Degenerate
- `order_id`, `line_id`

### Measures
- additive: `quantity`, `line_subtotal_excl_tax`, `discount_amount`
- non-additive: `unit_price_excl_tax`

### BTS attribution
- `is_campaign_attributed` (bool), `attribution_rule` from T9 bridge ∈ {PROMO_CODE, SESSION_BRIDGE, NULL}
- Rows attributed via SESSION_BRIDGE → `metric_certainty='INFERRED'`; otherwise CONFIRMED

In [ ]:
# fact_ecommerce_order_line (Lead). Grain: one row per EC order line (order_id, line_id).
lines  = read_silver('silver_ec_order_lines')
orders = read_silver('silver_ec_orders').select(
    'order_id', 'order_date', 'customer_id', 'delivery_method_code', 'pickup_store_id',
    'promo_code_applied', 'order_status')
bridge = read_silver('silver_campaign_attribution_bridge').select(
    'order_id', F.lit(True).alias('is_bridge'), 'attribution_rule').dropDuplicates(['order_id'])

f = (lines.join(orders, 'order_id', 'left')
          .join(bridge, 'order_id', 'left'))

f = (f
     # dim FKs (date + lead-owned dims real; member dims via natural-key surrogate)
     .withColumn('date_key',            date_key(F.col('order_date')))
     .withColumn('product_key',         surrogate_key(F.col('product_sku')))         # M3 dim (pending)
     .withColumn('customer_key',        surrogate_key(F.col('customer_id')))         # M2 dim (pending)
     .withColumn('channel_key',         surrogate_key(F.when(F.col('delivery_method_code') == 'BOPIS', F.lit('BOPIS')).otherwise(F.lit('EC'))))
     .withColumn('delivery_method_key', surrogate_key(F.col('delivery_method_code')))
     .withColumn('promotion_key',       surrogate_key(F.col('promo_code_applied')))
     .withColumn('store_key',           surrogate_key(F.col('pickup_store_id')))      # BOPIS pickup store; M4 dim (pending)
     # measures
     .withColumn('quantity',            F.col('quantity').cast('int'))
     .withColumn('line_subtotal_excl_tax', F.col('line_total_excl_tax').cast('double'))   # additive
     .withColumn('discount_amount',     F.col('discount_applied').cast('double'))          # additive
     .withColumn('unit_price_excl_tax', F.col('unit_price_excl_tax').cast('double'))       # NON-additive
     # BTS attribution + certainty
     .withColumn('is_campaign_attributed', F.coalesce(F.col('is_bridge'), F.lit(False)))
     .withColumn('attribution_rule',
                 F.when(F.col('promo_code_applied').isNotNull(), F.lit('PROMO_CODE'))
                  .when(F.col('is_bridge') == True, F.lit('SESSION_BRIDGE')).otherwise(F.lit(None))))

f = add_anomaly_columns(f)
f = flag(f, F.col('is_campaign_attributed') == True, 'ATTRIBUTION_SESSION_BRIDGE', 'CLEAN', 'INFERRED')

fact = f.select(
    'order_id', 'line_id',                                   # degenerate
    'date_key', 'product_key', 'customer_key', 'channel_key', 'delivery_method_key',
    'promotion_key', 'store_key',
    'product_sku', 'customer_id',                            # natural keys kept
    'quantity', 'line_subtotal_excl_tax', 'discount_amount', 'unit_price_excl_tax',
    'is_campaign_attributed', 'attribution_rule',
    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level')
write_gold(fact, 'fact_ecommerce_order_line')
assert_grain(fact, ['order_id', 'line_id'], 'fact_ecommerce_order_line')

## fact_return_line (M5)

### Grain declaration
> **TODO M5: write in your own words.**
> Note: one row per **returned product per return authorisation**. A single return can return multiple products → multiple rows.

### Dim FKs
- date_key (return_request_date AND return_receipt_date — role-played), product_key, customer_key, store_key (BORIS = original online channel), return_reason_key, channel_key (= original purchase channel), listing_condition_key (received condition)

### Degenerate
- `return_authorization_id`, `original_order_id`, `line_no`

### Measures (T8 — both period attributions)
- `quantity_returned`, `refund_amount`, `restocking_fee`
- `revenue_impact_original_period`, `revenue_impact_return_period`
- BORIS flag if returned at store but originally online

In [ ]:
# TODO M5

## fact_order_fulfilment (Lead) — Accumulating Snapshot — HARDEST FACT

### Grain declaration
> One row represents the full fulfilment lifecycle of a single EC order (accumulating snapshot). Grain enforced by `order_id`. **NULL milestone date FKs are EXPECTED** for orders that have not yet reached that milestone. Mixing in per-line rows would double-count the order-level lag measures.

### Pipeline milestones (role-played date FKs)
`placed_date_key` (order_date) · `captured_date_key` (MIN pg_transactions.captured_ts) · `shipped_date_key` (MIN PICKED_UP event) · `delivered_date_key` (MIN DELIVERED event) · `returned_date_key` (rr_return_requests).

### Lag measures (additive across orders, NOT across milestones)
`placed_to_captured_hours`, `picked_to_delivered_hours`, `total_lifecycle_hours`.

### Anomaly
A14 clock-drift orders propagate `DELIVERY_BEFORE_SHIP` / `metric_certainty='UNRELIABLE'`.

In [ ]:
# fact_order_fulfilment (Lead) — Accumulating Snapshot. Grain: order_id (one row per EC order).
# NULL milestone date FKs are EXPECTED for orders that have not reached that milestone.
orders = read_silver('silver_ec_orders').select(
    'order_id', 'order_number', 'order_date', 'order_status', 'customer_id')

# payment capture (Bronze pg_transactions; captured_ts = epoch seconds; order_ref = 'EC-...' = order_number)
pg = (read_bronze('pg_transactions').filter(F.col('captured_ts').isNotNull())
      .groupBy('order_ref').agg(F.min(F.col('captured_ts').cast('long')).alias('captured_epoch')))

# shipment milestones via dc_shipments(order_reference) -> silver_dc_delivery_events
ship = read_bronze('dc_shipments').select('shipment_id', F.col('order_reference').alias('order_number'))
ev   = read_silver('silver_dc_delivery_events').select('shipment_id', 'event_type_code', 'event_ts_parsed', 'anomaly_reason_code')
ev2  = ev.join(ship, 'shipment_id', 'inner')
picked    = ev2.filter(F.col('event_type_code') == 'PICKED_UP').groupBy('order_number').agg(F.min('event_ts_parsed').alias('picked_ts'))
delivered = ev2.filter(F.col('event_type_code') == 'DELIVERED').groupBy('order_number').agg(F.min('event_ts_parsed').alias('delivered_ts'))
drift     = (ev2.filter(F.col('anomaly_reason_code').contains('DELIVERY_BEFORE_SHIP'))
                .select('order_number').distinct().withColumn('a14', F.lit(True)))

# returns (Bronze rr_return_requests; return_request_date is DD-Mon-YYYY)
ret = (read_bronze('rr_return_requests')
       .withColumn('returned_dt', parse_date(F.col('return_request_date'), 'ddmonyyyy'))
       .groupBy(F.col('original_order_ref').alias('order_number')).agg(F.min('returned_dt').alias('returned_dt')))

f = (orders
     .join(pg, orders.order_number == pg.order_ref, 'left').drop('order_ref')
     .join(picked, 'order_number', 'left').join(delivered, 'order_number', 'left')
     .join(ret, 'order_number', 'left').join(drift, 'order_number', 'left'))

dk = lambda ts: F.when(ts.isNotNull(), date_key(F.date_format(ts, 'yyyy-MM-dd')))
f = (f
     .withColumn('placed_ts',   parse_date(F.col('order_date'), 'iso_date').cast('timestamp'))
     .withColumn('captured_ts', F.col('captured_epoch').cast('timestamp'))
     .withColumn('placed_date_key',    date_key(F.col('order_date')))
     .withColumn('captured_date_key',  dk(F.col('captured_ts')))
     .withColumn('shipped_date_key',   dk(F.col('picked_ts')))
     .withColumn('delivered_date_key', dk(F.col('delivered_ts')))
     .withColumn('returned_date_key',  dk(F.col('returned_dt').cast('timestamp')))
     .withColumn('placed_to_captured_hours',  (F.col('captured_ts').cast('long')  - F.col('placed_ts').cast('long')) / 3600.0)
     .withColumn('picked_to_delivered_hours', (F.col('delivered_ts').cast('long') - F.col('picked_ts').cast('long')) / 3600.0)
     .withColumn('total_lifecycle_hours',     (F.col('delivered_ts').cast('long') - F.col('placed_ts').cast('long')) / 3600.0)
     .withColumn('customer_key', surrogate_key(F.col('customer_id'))))

f = add_anomaly_columns(f)
f = flag(f, F.col('a14') == True, 'DELIVERY_BEFORE_SHIP', 'FLAGGED_ANOMALY', 'UNRELIABLE')

fact = f.select(
    'order_id', 'order_number', 'customer_key', 'customer_id', 'order_status',           # degenerate / nat keys
    'placed_date_key', 'captured_date_key', 'shipped_date_key', 'delivered_date_key', 'returned_date_key',
    'placed_to_captured_hours', 'picked_to_delivered_hours', 'total_lifecycle_hours',
    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level')
write_gold(fact, 'fact_order_fulfilment')
assert_grain(fact, ['order_id'], 'fact_order_fulfilment')   # accumulating snapshot = single grain

---
# Inventory Group (M4)

## fact_store_inventory_snapshot (Lead)

### Grain declaration
> One row represents the inventory position of one SKU in one store on one calendar day (periodic snapshot). Grain enforced by (`store_id`, `product_code`, `snapshot_date`). Quantities are **semi-additive** — valid to sum across SKU/store on a single date, **invalid across dates**. UNIONs M4's clean snapshots with the T6-reconstructed sibling.

### Dim FKs
- date_key (snapshot_date), product_key, store_key

### Measures (semi-additive)
- `physical_qty`, `sellable_qty`, `reserved_qty`, `damaged_qty`

### Reconstructed rows
- Carry `data_quality_status='RECONSTRUCTED'`, `metric_certainty='INFERRED'` from T6; clean rows are CONFIRMED.

In [ ]:
# fact_store_inventory_snapshot (Lead). Grain: store_id x product_code x snapshot_date (periodic, semi-additive).
# UNION of M4's clean snapshots (read from Bronze while M4 Silver is pending; CONFIRMED) + the T6
# reconstructed sibling (INFERRED). ATP/qty are semi-additive: sum across SKU/store on ONE date, NOT across dates.
clean = (read_bronze('si_inventory_snapshots')
         .select('store_id', 'product_code', 'snapshot_date',
                 F.col('physical_qty').cast('int').alias('physical_qty'),
                 F.col('sellable_qty').cast('int').alias('sellable_qty'),
                 F.col('reserved_qty').cast('int').alias('reserved_qty'),
                 F.col('damaged_qty').cast('int').alias('damaged_qty'))
         .withColumn('is_reconstructed', F.lit(False)))
clean = add_anomaly_columns(clean)   # CONFIRMED / CLEAN

recon = (read_silver('silver_store_inventory_snapshots_reconstructed')
         .select('store_id', 'product_code', 'snapshot_date', 'physical_qty', 'sellable_qty',
                 F.lit(None).cast('int').alias('reserved_qty'), F.lit(None).cast('int').alias('damaged_qty'),
                 'is_reconstructed',
                 'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))

both = clean.unionByName(recon)
fact = (both
        .withColumn('snapshot_date_iso', F.date_format(parse_date(F.col('snapshot_date'), 'ddmmyyyy'), 'yyyy-MM-dd'))
        .withColumn('date_key',    date_key(F.col('snapshot_date_iso')))
        .withColumn('product_key', surrogate_key(F.col('product_code')))
        .withColumn('store_key',   surrogate_key(F.col('store_id'))))
write_gold(fact, 'fact_store_inventory_snapshot')
assert_grain(fact, ['store_id', 'product_code', 'snapshot_date'], 'fact_store_inventory_snapshot')
print('reconstructed rows in fact = %d'
      % fact.filter(F.col('metric_certainty_level') == 'INFERRED').count())

## fact_warehouse_inventory_snapshot (M4)

### Grain declaration
> **TODO M4: write in your own words.**
> Periodic: SKU × warehouse × calendar day.

### Same shape as store but warehouse_key replaces store_key. Warehouses are gapless — no reconstructed rows.
Includes `in_transit_qty`, `inbound_pending_qty`.

In [ ]:
# TODO M4

## fact_inventory_transaction (Lead)

### Grain declaration
> One row represents a single inventory movement event. Grain enforced by (`movement_id`, `node_type`) — store and warehouse movement IDs share a namespace, so the node type disambiguates. `quantity_delta` is additive and already signed (RCVD/RET +, PICK/DMG −).

### Dim FKs
- date_key (movement_date), product_key, node_key (store or warehouse)

### Degenerate
- `movement_id`, `reference_number` (NULL for B3 cases — flagged `MOVEMENT_NULL_REF`)

### Measures
- additive: `quantity_delta`

In [ ]:
# fact_inventory_transaction (Lead). Grain: one row per inventory movement (movement_id x node_type).
# quantity_delta is additive and already signed (RCVD/RET +, PICK/DMG -). Store + warehouse movements unioned.
si = (read_bronze('si_inventory_movements')
      .select(F.col('movement_id'), F.lit('STORE').alias('node_type'),
              F.col('store_id').cast('string').alias('node_id'), F.col('product_code').alias('product_code'),
              F.col('movement_type'), F.col('quantity_change').cast('int').alias('quantity_delta'),
              F.col('reference_number'),
              parse_date(F.col('movement_date'), 'ddmmyyyy').alias('mdate')))
wh = (read_bronze('wh_inventory_movements')
      .select(F.col('movement_id'), F.lit('WAREHOUSE').alias('node_type'),
              F.col('warehouse_id').cast('string').alias('node_id'), F.col('sku').alias('product_code'),
              F.col('movement_type'), F.col('quantity_change').cast('int').alias('quantity_delta'),
              F.col('reference_number'),
              F.to_date(F.col('movement_datetime')).alias('mdate')))
mv = si.unionByName(wh)
fact = (add_anomaly_columns(mv)
        .withColumn('date_key',    date_key(F.date_format(F.col('mdate'), 'yyyy-MM-dd')))
        .withColumn('product_key', surrogate_key(F.col('product_code')))
        .withColumn('node_key',    surrogate_key(F.col('node_type'), F.col('node_id'))))
# B3: warehouse picks with NULL reference (175) carried through as ambiguous
fact = flag(fact, F.col('reference_number').isNull() & (F.col('node_type') == 'WAREHOUSE'),
            'MOVEMENT_NULL_REF', 'FLAGGED_AMBIGUOUS', 'INFERRED')
write_gold(fact, 'fact_inventory_transaction')
assert_grain(fact, ['movement_id', 'node_type'], 'fact_inventory_transaction')

---
# Digital Group (Lead)

## fact_web_session (Lead)

### Grain declaration
> One row represents a single web session. Grain enforced by `session_id`. Mixing in page events would break additivity for `session_duration_seconds` and `page_count`.

### Dim FKs
- date_key (session_start), customer_key (NULL for unauthenticated), promotion_key (from UTM)

### Degenerate
- `session_id`

### Measures
- `session_duration_seconds`, `page_count`, `event_count`

In [ ]:
# fact_web_session (Lead). Grain: one row per web session (session_id).
s = read_silver('silver_ws_sessions')
fact = (s
        .withColumn('date_key', F.when(F.col('session_start_parsed').isNotNull(),
                                        date_key(F.date_format(F.col('session_start_parsed'), 'yyyy-MM-dd'))))
        .withColumn('customer_key', F.when(F.col('registered_customer_id').isNotNull(),
                                           surrogate_key(F.col('registered_customer_id'))))
        .withColumn('promotion_key', surrogate_key(F.col('utm_campaign')))
        .withColumn('session_duration_seconds',
                    (F.col('session_end_parsed').cast('long') - F.col('session_start_parsed').cast('long')))
        .select('session_id', 'date_key', 'customer_key', 'promotion_key',
                'registered_customer_id', 'utm_campaign', 'device_type', 'channel_source',
                'session_duration_seconds', F.col('total_pageviews').cast('int').alias('page_count'),
                F.col('total_events').cast('int').alias('event_count'),
                'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
write_gold(fact, 'fact_web_session')
assert_grain(fact, ['session_id'], 'fact_web_session')

## fact_web_page_event (Lead) — highest-volume fact

### Grain declaration
> One row represents a single page event within a session (the highest-volume fact, ~21K rows). Grain enforced by (`session_id`, `event_id`).

### Dim FKs
- date_key (event_datetime), step_key (funnel step), product_key (NULL if non-PDP)

### Degenerate
- `session_id`, `event_id`

### Measures
- `time_on_page_seconds`, `cart_value`, `event_type_code`, `is_conversion_event`

In [ ]:
# fact_web_page_event (Lead) — highest-volume fact (~21K). Grain: (session_id, event_id).
pe = read_silver('silver_ws_page_events')
fact = (pe
        .withColumn('date_key', F.when(F.col('event_ts_parsed').isNotNull(),
                                        date_key(F.date_format(F.col('event_ts_parsed'), 'yyyy-MM-dd'))))
        .withColumn('product_key', F.when(F.col('product_sku').isNotNull(), surrogate_key(F.col('product_sku'))))
        .select('session_id', 'event_id', 'date_key', 'product_key',
                'event_type_code', 'page_category', 'product_sku', 'search_query', 'step_id',
                F.col('time_on_page_secs').cast('int').alias('time_on_page_seconds'),
                F.col('cart_value').cast('double').alias('cart_value'), 'is_conversion_event',
                'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
write_gold(fact, 'fact_web_page_event')
assert_grain(fact, ['session_id', 'event_id'], 'fact_web_page_event')

---
# NexaLocal / Marketplace / Seller / CX Group (M6)

## fact_classified_listing_event (M6)

### Grain declaration
> **TODO M6: write in your own words.**
> One row per engagement event on a NexaLocal listing.

### Dim FKs
- date_key (event_datetime), product_key (canonical via T7 attempt; NULL if unmatched), seller_key, customer_key (the engaging user)

### Degenerate
- `listing_ref`, `event_id`

### Measures
- event_type (Viewed/Favourited/ChatInitiated/PhoneRevealed/OfferMade/SellerSold)
- listing_asking_price (at event time)
- listing_confidence_score (from T7)

### Metric certainty
- CONFIRMED for platform engagement events
- **INFERRED** for SELLER_SOLD (does NOT confirm transaction — A4)

In [ ]:
# TODO M6

## fact_classified_listing_snapshot (Lead)

### Grain declaration
> One row represents the current state of one NexaLocal listing. Grain enforced by `listing_id`. `asking_price` is non-additive. (Engagement events are the separate `fact_classified_listing_event`, M6's.)

### Dim FKs
- date_key (created date), product_key (category), seller_key

### Degenerate
- `listing_ref`

### Measures
- `views_count`, `favourites_count` (additive); `asking_price` (non-additive); `relist_count`
- A12 `RELISTED_AFTER_SOLD` flag carried from Silver.

In [ ]:
# fact_classified_listing_snapshot (Lead). Grain: one row per NL listing (listing state snapshot).
# Source nl_listings is listing-grain (no daily snapshot source exists), so grain = listing_id; carries the
# A12 RELISTED_AFTER_SOLD flag from Silver. asking_price is non-additive.
nl = read_silver('silver_nl_listings')
fact = (nl
        .withColumn('date_key', F.when(F.col('created_at_parsed').isNotNull(),
                                        date_key(F.date_format(F.col('created_at_parsed'), 'yyyy-MM-dd'))))
        .withColumn('product_key', surrogate_key(F.col('category_code')))     # M3 dim (pending)
        .withColumn('seller_key',  surrogate_key(F.col('seller_account_id')))  # M6 dim (pending)
        .select('listing_id', 'listing_ref', 'date_key', 'product_key', 'seller_key',
                'seller_account_id', 'category_code', 'status_code',
                F.col('asking_price').cast('double').alias('asking_price'),
                F.col('views_count').cast('int').alias('views_count'),
                F.col('favourites_count').cast('int').alias('favourites_count'),
                F.col('relist_count').cast('int').alias('relist_count'),
                'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level'))
write_gold(fact, 'fact_classified_listing_snapshot')
assert_grain(fact, ['listing_id'], 'fact_classified_listing_snapshot')

## fact_seller_performance_snapshot (M2) — Periodic per seller per week

### Grain declaration
> **TODO M2: write in your own words.**
> One row per seller per ISO week.

### Dim FKs
- date_key (week_end_date), seller_key, seller_risk_tier_key

### Measures — store NUMERATORS and DENOMINATORS, NOT pre-divided ratios
For each rate metric (cancellation, late_fulfilment, return, complaint, nl_duplicate, nl_report, response):
- `<metric>_numerator` (e.g. `cancelled_orders`)
- `<metric>_denominator` (e.g. `total_orders`)
Marts compute `metric_rate = numerator / NULLIF(denominator, 0)` at query time. This way SUMs work correctly across weeks.

Plus the composite `seller_trust_score` (non-additive — store, marts handle aggregation).

In [ ]:
# TODO M6

## fact_customer_review (M6)

### Grain declaration
> **TODO M6: write in your own words.**
> One row per review.

### Dim FKs
- date_key (review_datetime), product_key, customer_key, store_key (NULL if not store-related), seller_key (NULL if first-party product), seller_risk_tier_key

### Degenerate
- `review_id`

### Measures
- star_rating (1–5; non-additive — average at mart time)
- review_type, is_verified_purchase, days_post_delivery
- A15 flagged rows have negative `days_post_delivery`

In [ ]:
# TODO M6

## fact_customer_complaint (Lead)

### Grain declaration
> One row represents a single support case. Grain enforced by `case_id`; `canonical_case_key` is the dedup column so marts can collapse the A16 duplicate cases (rows with `is_duplicate_flag=1` carry their canonical ref).

### Dim FKs
- date_key (case open), customer_key, channel_key (attributed contact channel)

### Degenerate
- `canonical_case_key` (from `canonical_case_ref`), `case_reference`

### Measures
- `resolution_time_hours`, `resolution_outcome`, `fault_attribution`, `is_duplicate_flag`
- A16 `DUPLICATE_CASE` flagged where `is_duplicate_flag=1` (expect 7).

In [ ]:
# fact_customer_complaint (Lead). Grain: one row per support case (case_id); canonical_case_key is the
# dedup column (A16: is_duplicate_flag=1 rows carry their canonical ref so marts collapse them).
# cs_cases datetimes are YYYY/MM/DD HH:MM (slash_datetime).
cs = read_bronze('cs_cases')
fact = (cs
        .withColumn('created_dt',  parse_date(F.col('created_datetime'),  'slash_datetime'))
        .withColumn('resolved_dt', parse_date(F.col('resolved_datetime'), 'slash_datetime'))
        .withColumn('canonical_case_key', F.coalesce(F.col('canonical_case_ref'), F.col('case_reference')))
        .withColumn('date_key', F.when(F.col('created_dt').isNotNull(),
                                       date_key(F.date_format(F.col('created_dt'), 'yyyy-MM-dd'))))
        .withColumn('customer_key', surrogate_key(F.col('customer_ref')))            # M2 dim (pending)
        .withColumn('channel_key',  surrogate_key(F.col('channel_attribution')))
        .withColumn('resolution_time_hours',
                    (F.col('resolved_dt').cast('long') - F.col('created_dt').cast('long')) / 3600.0))
fact = add_anomaly_columns(fact)
fact = flag(fact, F.col('is_duplicate_flag') == 1, 'DUPLICATE_CASE', 'FLAGGED_ANOMALY', 'INFERRED')
fact = fact.select(
    'case_id', 'canonical_case_key', 'case_reference', 'date_key', 'customer_key', 'channel_key',
    'customer_ref', 'category_code', 'status_code', 'resolution_outcome', 'fault_attribution',
    'resolution_time_hours', 'is_duplicate_flag',
    'anomaly_flag', 'anomaly_reason_code', 'data_quality_status', 'metric_certainty_level')
write_gold(fact, 'fact_customer_complaint')
assert_grain(fact, ['case_id'], 'fact_customer_complaint')
print('A16 DUPLICATE_CASE flagged = %d (expect 7)'
      % fact.filter(F.col('anomaly_reason_code').contains('DUPLICATE_CASE')).count())

---
## Acceptance for the Gold fact layer

Lead checks:
1. All 14 facts exist in `NEXAMART_GOLD`
2. **Every fact has a written grain declaration in markdown above its code cell** — in the owner's own words, not copied from this scaffold's suggestions
3. `assert_grain()` passes for every fact
4. All FKs are surrogate keys, never operational natural keys
5. `metric_certainty_level` populated on every row of every fact
6. Degenerate dim columns present (receipt_no, order_id, return_auth_id, listing_ref, canonical_case_key)
7. Rate metrics in `fact_seller_performance_snapshot` stored as numerator/denominator pairs